# Goal: statistical-summary 

In [1]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\n'

In [2]:
import polars as pl

In [3]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-15\data\lev-15_merged.parquet"
pdf = pl.scan_parquet(path)

In [4]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('VISIT', String),
        ('LEVEL', String),
        ('SECTION', String),
        ('TIME_TAKEN', String),
        ('MONTHLY_CONSUMPTION_EXP', Float64),
        ('ONLINE_EXPENDITURE', Float64),
        ('INFORMANT_CODE', String),
        ('RESPONSE_CODE', String),
        ('HOUSEHOLD_SIZE', Float64),
        ('VISIT_MONTH', Int64),
        ('MULTIPLIER', Int64)])

# Useful Variables

In [5]:

cols = ['FSU_Serial_No',
    'TIME_TAKEN',
 'MONTHLY_CONSUMPTION_EXP',
 'ONLINE_EXPENDITURE',
 'INFORMANT_CODE',
 'RESPONSE_CODE',
 'HOUSEHOLD_SIZE',
 'VISIT_MONTH',]

In [6]:
df = pdf.select(cols)

In [7]:
df.head(2).collect()

FSU_Serial_No,TIME_TAKEN,MONTHLY_CONSUMPTION_EXP,ONLINE_EXPENDITURE,INFORMANT_CODE,RESPONSE_CODE,HOUSEHOLD_SIZE,VISIT_MONTH
str,str,f64,f64,str,str,f64,i64
"""46956""","""15""",null,null,"""""","""""",null,112023
"""46956""","""50""",7000.0,0.0,"""4""","""1""",7.0,112023


In [8]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in cols]
)

In [9]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

FSU_Serial_No,TIME_TAKEN,MONTHLY_CONSUMPTION_EXP,ONLINE_EXPENDITURE,INFORMANT_CODE,RESPONSE_CODE,HOUSEHOLD_SIZE,VISIT_MONTH
u32,u32,u32,u32,u32,u32,u32,u32
14827,91,5480,4301,23,6,29,13


# Logic

In [10]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_10100\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [11]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
FSU_Serial_No,2095624.0,0.0,37348.497543,9904.29055,22300.0,26451.0,42231.0,46160.0,49999.0
TIME_TAKEN,2095544.0,80.0,39.5672,10.022711,10.0,32.0,42.0,49.0,150.0
MONTHLY_CONSUMPTION_EXP,1571718.0,523906.0,12894.407002,7653.674097,200.0,8000.0,11500.0,16000.0,155000.0
ONLINE_EXPENDITURE,1571718.0,523906.0,598.123904,1946.26921,0.0,0.0,0.0,299.0,100000.0
INFORMANT_CODE,1571718.0,523906.0,2.213434,7.060532,1.0,1.0,2.0,2.0,99.0
HOUSEHOLD_SIZE,1571718.0,523906.0,4.219136,2.072564,1.0,3.0,4.0,5.0,31.0
VISIT_MONTH,2095624.0,0.0,62515.462393,35875.146619,12024.0,32024.0,52024.0,92023.0,122023.0


# Categorical Columns

In [12]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))

RESPONSE_CODE


RESPONSE_CODE,count
i32,u32
1,1381214
null,523906
2,152870
3,25780
4,10630
9,1224
